In [ ]:
# Install minimal required libraries
!pip install -q torch transformers accelerate

import os
import sys
import json
import time
import random
import numpy as np
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM

# 1. Reproducibility & Folder Setup
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

os.makedirs("results", exist_ok=True)

print("=== 1. ENVIRONMENT & HARDWARE DIAGNOSTICS ===")
print(f"Python Version       : {sys.version.split()[0]}")
print(f"PyTorch Version      : {torch.__version__}")
print(f"Transformers Version : {transformers.__version__}")
print(f"CUDA Available       : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Device           : {torch.cuda.get_device_name(0)} ({vram_gb:.2f} GB VRAM)")
else:
    print("WARNING: Running on CPU. Switch Runtime -> T4 GPU for faster execution.")

# 2. Load Compact Open-Weights Generator (Fits easily in 16 GB T4 VRAM in float16)
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
print(f"\nLoading {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

# 3. Controlled Academic Passages Testing H1 (Qualifiers) & H4 (Contradictions)
# Paper A contains a headline claim PLUS a critical mid-document qualifier (H1).
paper_a = (
    "Chen et al. (2024) evaluated SparseRAG on multi-hop scientific QA and reported an "
    "84.2% F1 headline score. However, their ablation study revealed a critical limitation: "
    "gains occurred ONLY on corpora under 5,000 tokens, while accuracy degraded by 12.4% on "
    "longer multi-paper collections due to missed cross-references."
)

# Paper B directly contradicts Paper A's effectiveness claim (H4).
paper_b = (
    "Patel and Gomez (2025) replicated SparseRAG across three biomedical review benchmarks "
    "and reported contradictory findings: SparseRAG showed no statistically significant gain "
    "over standard BM25 retrieval (p = 0.41) and increased end-to-end latency by 2.3x."
)

# Topical distractors sharing similar academic vocabulary (H3 distractor context)
distractors = [
    "Alvarez et al. (2023) benchmarked dense bi-encoder retrievers on PubMedQA and noted that chunk overlap of 128 tokens improved recall by 3.1%.",
    "Kim et al. (2024) studied citation formatting in automated biomedical summaries and found that inline bracketed IDs reduced sentence-level drift.",
    "O'Connor et al. (2024) analyzed token-budget allocation in retrieval pipelines and recommended capping context windows at 8,192 tokens for latency-sensitive tasks."
]

def run_synthesis(docs_list, condition_name):
    formatted_docs = "\n\n".join([f"[Doc {i+1}] {text}" for i, text in enumerate(docs_list)])
    messages = [
        {
            "role": "system",
            "content": (
                "You are an academic literature synthesis assistant. Synthesize the findings of the "
                "provided papers in 3 to 4 sentences. Cite sources as [Doc X]. Explicitly report any "
                "methodological limitations, scope conditions, or conflicting results between papers."
            )
        },
        {
            "role": "user",
            "content": (
                f"Papers:\n{formatted_docs}\n\n"
                "Task: Synthesize the evidence on SparseRAG's effectiveness and limitations."
            )
        }
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    t0 = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=180,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    elapsed = round(time.time() - t0, 2)
    gen_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    # Automated sanity check for H1 (qualifier retention) and H4 (contradiction reporting)
    lower_out = gen_text.lower()
    h1_qualifier_kept = any(k in lower_out for k in ["5,000", "5000", "degraded", "12.4%", "limit"])
    h4_conflict_kept = any(k in lower_out for k in ["contradict", "conflict", "no statistically", "however", "disagree", "unlike"])

    print(f"\n--- {condition_name} (Time: {elapsed}s) ---")
    print(gen_text)
    print(f"[Auto-Check] H1 Qualifier Retained: {h1_qualifier_kept} | H4 Contradiction Reported: {h4_conflict_kept}")

    return {
        "condition": condition_name,
        "latency_sec": elapsed,
        "output": gen_text,
        "h1_qualifier_retained": h1_qualifier_kept,
        "h4_contradiction_reported": h4_conflict_kept
    }

# Condition 1: Target papers placed at START ([Doc 1]) and END ([Doc 5])
docs_edges = [paper_a, distractors[0], distractors[1], distractors[2], paper_b]

# Condition 2: Target papers buried in the MIDDLE ([Doc 2] and [Doc 3])
docs_middle = [distractors[0], paper_a, paper_b, distractors[1], distractors[2]]

res_edges = run_synthesis(docs_edges, "CONDITION A: Target Papers at START & END")
res_middle = run_synthesis(docs_middle, "CONDITION B: Target Papers in MIDDLE")

# Save genuine execution log to results/
summary = {
    "model": MODEL_ID,
    "seed": SEED,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "runs": [res_edges, res_middle]
}
with open("results/week1_sanity_output.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\n=== SAVED GENUINE OUTPUT TO results/week1_sanity_output.json ===")

In [ ]:
!pip install -q torch transformers datasets sentence-transformers accelerate matplotlib pandas

import os
import re
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, CrossEncoder, util
from transformers import AutoTokenizer, AutoModelForCausalLM

# =====================================================================
# 1. FROZEN CONFIGURATION & DETERMINISTIC SEEDS
# =====================================================================
SEED = 42
N_CLUSTERS = 5          # Run 5 first for Gate G2 verification; set to 20 for full Gate G3 run
CORPUS_SIZES = [5, 15, 25]
POSITIONS = ["start", "middle", "end"]

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
RETRIEVER_ID = "BAAI/bge-small-en-v1.5"
NLI_ID = "cross-encoder/nli-deberta-v3-small"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

os.makedirs("results", exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on device: {device} | N_CLUSTERS={N_CLUSTERS} | K={CORPUS_SIZES}")

# =====================================================================
# 2. LOAD REAL SCIENTIFIC ABSTRACTS (qiaojin/PubMedQA)
# =====================================================================
print("Loading real scientific abstracts from HuggingFace (qiaojin/PubMedQA)...")
ds = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")

yes_records = [row for row in ds if row["final_decision"] == "yes" and len(row["context"]["contexts"]) >= 2]
no_records = [row for row in ds if row["final_decision"] == "no" and len(row["context"]["contexts"]) >= 2]
distractor_pool = [
    " ".join(row["context"]["contexts"])[:1100]
    for row in ds
    if row["final_decision"] == "maybe" and len(" ".join(row["context"]["contexts"])) > 400
]

print(f"Loaded {len(yes_records)} positive studies, {len(no_records)} negative studies, {len(distractor_pool)} topical distractors.")

def build_factorial_clusters(n_clusters: int, seed: int = 42):
    rng = random.Random(seed)
    clusters = []
    for idx in range(n_clusters):
        pos_study = yes_records[idx % len(yes_records)]
        neg_study = no_records[idx % len(no_records)]

        topic_question = pos_study["question"]
        # Split positive study so a realistic methodological qualifier sits in the MIDDLE of the paper (Tests H1)
        ctx_parts = pos_study["context"]["contexts"]
        intro_part = ctx_parts[0][:380]
        result_part = " ".join(ctx_parts[1:])[:420]

        h1_qualifier_phrase = "subgroup under 65 years without renal impairment"
        paper_positive = (
            f"Study A ({pos_study['pubid']}): Background and Design: {intro_part} "
            f"Methodological Qualifier: Stratified analysis revealed that treatment efficacy held ONLY in the "
            f"{h1_qualifier_phrase}, whereas benefit declined in older or comorbid patients. "
            f"Primary Outcome: {result_part}"
        )

        # Contradicting negative study on the same research question (Tests H4)
        neg_body = " ".join(neg_study["context"]["contexts"])[:550]
        paper_contradicting = (
            f"Study B ({neg_study['pubid']}): Contradictory Replication: Contrary to positive single-center reports on "
            f"'{topic_question}', this multi-center trial found no statistically significant clinical benefit "
            f"over standard control (p = 0.43) and observed higher adverse-event discontinuation. Details: {neg_body}"
        )

        # Sample up to 23 distinct domain-matched distractor abstracts (Tests H3)
        sampled_distractors = rng.sample(distractor_pool, k=max(CORPUS_SIZES) - 2)

        clusters.append({
            "cluster_id": f"C{idx+1:02d}",
            "question": topic_question,
            "paper_pos": paper_positive,
            "paper_neg": paper_contradicting,
            "distractors": sampled_distractors,
            "h1_keywords": ["under 65", "renal impairment", "subgroup", "only in"],
            "h4_keywords": ["contrary", "contradict", "no statistically significant", "p = 0.43", "however", "conflict", "disagree"],
            "headline_keywords": [w.lower() for w in pos_study["long_answer"].split() if len(w) > 6][:5]
        })
    return clusters

def assemble_prompt_docs(cluster: dict, k: int, position: str):
    """Places [paper_pos, paper_neg] at Start, Middle, or End amidst k-2 distractors."""
    sub_distractors = list(cluster["distractors"][:k - 2])
    target_pair = [cluster["paper_pos"], cluster["paper_neg"]]

    if position == "start":
        ordered = target_pair + sub_distractors
    elif position == "end":
        ordered = sub_distractors + target_pair
    elif position == "middle":
        mid = len(sub_distractors) // 2
        ordered = sub_distractors[:mid] + target_pair + sub_distractors[mid:]
    else:
        raise ValueError(f"Invalid position: {position}")
    return ordered

# Unit test verifying exact document count and placement
_test_c = build_factorial_clusters(1)[0]
for _p in POSITIONS:
    _d = assemble_prompt_docs(_test_c, 15, _p)
    assert len(_d) == 15
    if _p == "start":
        assert _d[0] == _test_c["paper_pos"]
    elif _p == "end":
        assert _d[-1] == _test_c["paper_neg"]
    elif _p == "middle":
        assert _d[6] == _test_c["paper_pos"]
print("[PASS] Unit test verified: Factorial cluster assembly (Start / Middle / End).")

# =====================================================================
# 3. INITIALIZE MODELS (GENERATOR, RETRIEVER, NLI VERIFIER)
# =====================================================================
print(f"Loading Retriever ({RETRIEVER_ID}) and NLI Verifier ({NLI_ID})...")
retriever = SentenceTransformer(RETRIEVER_ID, device=device)
nli_verifier = CrossEncoder(NLI_ID, device=device)

print(f"Loading Generator ({MODEL_ID})...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
llm = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto"
)
llm.eval()

def call_llm_synthesis(question: str, indexed_docs: list) -> str:
    context_block = "\n\n".join([f"[Doc {doc_num}] {text}" for doc_num, text in indexed_docs])
    messages = [
        {
            "role": "system",
            "content": (
                "You are a rigorous academic synthesis system. Synthesize the findings relevant to the "
                "research question using ONLY the provided documents. Cite every claim with [Doc X]. "
                "Explicitly state any subgroup/methodological qualifiers and report any conflicting results."
            )
        },
        {
            "role": "user",
            "content": f"Documents:\n{context_block}\n\nResearch Question: {question}\n\nAttributed Synthesis:"
        }
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(llm.device)
    with torch.inference_mode():
        out = llm.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    if device == "cuda":
        torch.cuda.empty_cache()
    return text

# =====================================================================
# 4. THREE SYSTEMS: BASELINE 1, BASELINE 2, PROPOSED (FIGURE 2)
# =====================================================================
def run_baseline1_full_context(question: str, docs: list):
    t0 = time.time()
    indexed = [(i + 1, d) for i, d in enumerate(docs)]
    ans = call_llm_synthesis(question, indexed)
    return ans, time.time() - t0

def run_baseline2_dense_rag(question: str, docs: list, top_k: int = 4):
    t0 = time.time()
    q_emb = retriever.encode(question, convert_to_tensor=True)
    d_embs = retriever.encode(docs, convert_to_tensor=True)
    scores = util.cos_sim(q_emb, d_embs)[0]
    top_idx = torch.topk(scores, k=min(top_k, len(docs))).indices.tolist()
    indexed = [(i + 1, docs[i]) for i in top_idx]
    ans = call_llm_synthesis(question, indexed)
    return ans, time.time() - t0

def run_proposed_figure2(question: str, docs: list):
    """Operationalizes Review Figure 2: Edge-Salience Reordering + NLI Claim Verification."""
    t0 = time.time()
    q_emb = retriever.encode(question, convert_to_tensor=True)
    d_embs = retriever.encode(docs, convert_to_tensor=True)
    scores = util.cos_sim(q_emb, d_embs)[0]
    ranked = torch.argsort(scores, descending=True).tolist()

    # Stage A: Place top-ranked documents at primacy (start) and recency (end) edges
    left, right = [], []
    for r, idx in enumerate(ranked):
        if r % 2 == 0:
            left.append(idx)
        else:
            right.insert(0, idx)
    reordered = left + right
    indexed = [(i + 1, docs[i]) for i in reordered]
    draft = call_llm_synthesis(question, indexed)

    # Stage B: NLI Claim-Level Verification against top-2 relevant studies
    top1, top2 = ranked[0], ranked[1]
    pairs = [(docs[top1], draft), (docs[top2], draft)]
    logits = nli_verifier.predict(pairs)
    entail_probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].tolist()

    # If either key paper is omitted/unentailed, run focused verification synthesis
    if min(entail_probs) < 0.45:
        focused = [(top1 + 1, docs[top1]), (top2 + 1, docs[top2])]
        final_ans = call_llm_synthesis(question, focused)
        return final_ans, time.time() - t0
    return draft, time.time() - t0

# =====================================================================
# 5. AUTOMATED EVALUATION (H1 QUALIFIERS, H3 ATTRIBUTION, H4 CONFLICTS)
# =====================================================================
def score_output(output_text: str, docs: list, cluster: dict) -> dict:
    low = output_text.lower()

    # Headline coverage (%)
    h_hits = sum(1 for kw in cluster["headline_keywords"] if kw in low)
    headline_pct = (h_hits / max(1, len(cluster["headline_keywords"]))) * 100.0

    # H1: Mid-document qualifier retention (binary 100/0)
    h1_retained = 100.0 if any(kw in low for kw in cluster["h1_keywords"]) else 0.0

    # H4: Contradiction detection (binary 100/0)
    h4_detected = 100.0 if any(kw in low for kw in cluster["h4_keywords"]) else 0.0

    # H3: Citation Attribution Precision via DeBERTa-v3 NLI
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', output_text) if s.strip()]
    cited_total, entailed_total = 0, 0
    for sent in sentences:
        cids = re.findall(r'\[Doc\s*(\d+)\]', sent)
        if cids:
            cited_total += 1
            premises = [docs[int(c) - 1] for c in cids if 1 <= int(c) <= len(docs)]
            if premises:
                logits = nli_verifier.predict([(" ".join(premises), sent)])
                prob = torch.softmax(torch.tensor(logits), dim=-1)[0, 1].item()
                if prob >= 0.45:
                    entailed_total += 1
    faithfulness_pct = (entailed_total / cited_total * 100.0) if cited_total > 0 else 0.0

    return {
        "headline_recall_pct": round(headline_pct, 2),
        "h1_qualifier_recall_pct": h1_retained,
        "h4_contradiction_pct": h4_detected,
        "h3_nli_faithfulness_pct": round(faithfulness_pct, 2)
    }

# =====================================================================
# 6. RUN FACTORIAL SWEEP & EXPORT CSV + PUBLICATION PDF FIGURES
# =====================================================================
clusters = build_factorial_clusters(N_CLUSTERS, seed=SEED)
records = []

total_runs = len(clusters) * len(CORPUS_SIZES) * len(POSITIONS) * 3
step = 0

for cluster in clusters:
    for k in CORPUS_SIZES:
        for pos in POSITIONS:
            docs = assemble_prompt_docs(cluster, k, pos)
            systems = [
                ("Baseline_1_FullContext", run_baseline1_full_context),
                ("Baseline_2_DenseRAG", run_baseline2_dense_rag),
                ("Proposed_Fig2_Verified", run_proposed_figure2),
            ]
            for sys_name, fn in systems:
                step += 1
                ans, lat = fn(cluster["question"], docs)
                metrics = score_output(ans, docs, cluster)
                records.append({
                    "cluster_id": cluster["cluster_id"],
                    "k_docs": k,
                    "position": pos,
                    "system": sys_name,
                    "latency_sec": round(lat, 2),
                    **metrics,
                    "generated_text": ans
                })
                if step % 9 == 0 or step == total_runs:
                    print(f"Progress: {step}/{total_runs} runs completed...")

df = pd.DataFrame(records)
df.to_csv("results/experiment_results.csv", index=False)

summary = df.groupby(["system", "k_docs", "position"])[
    ["headline_recall_pct", "h1_qualifier_recall_pct", "h4_contradiction_pct", "h3_nli_faithfulness_pct", "latency_sec"]
].mean().round(2).reset_index()
summary.to_csv("results/summary_metrics.csv", index=False)

print("\n=== SUMMARY METRICS TABLE (REAL EXECUTION OUTPUT) ===")
print(summary.to_string(index=False))

# Plot 1: Positional Degradation Curve (Start vs Middle vs End at max K)
max_k = max(CORPUS_SIZES)
sub_max = summary[summary["k_docs"] == max_k]
plt.figure(figsize=(7.5, 4.5))
for sys_name, marker in [("Baseline_1_FullContext", "o"), ("Baseline_2_DenseRAG", "s"), ("Proposed_Fig2_Verified", "^")]:
    s_df = sub_max[sub_max["system"] == sys_name].set_index("position").loc[POSITIONS]
    plt.plot(POSITIONS, s_df["h1_qualifier_recall_pct"], marker=marker, linewidth=2, label=sys_name)

plt.title(f"Mid-Document Qualifier Retention (H1) by Position at k={max_k} Papers")
plt.xlabel("Target Paper Position in Prompt")
plt.ylabel("H1 Qualifier Retention (%)")
plt.ylim(-5, 105)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.savefig("results/fig1_positional_degradation.pdf")
plt.savefig("results/fig1_positional_degradation.png", dpi=300)

# Plot 2: Overall System Comparison Across All Conditions
overall = df.groupby("system")[
    ["h1_qualifier_recall_pct", "h4_contradiction_pct", "h3_nli_faithfulness_pct"]
].mean()
ax = overall.plot(kind="bar", figsize=(8, 4.5), rot=0)
plt.title("Overall Synthesis Reliability Across Systems (H1, H4, H3)")
plt.ylabel("Score (%)")
plt.ylim(0, 110)
plt.legend(["H1 Qualifier Recall", "H4 Contradiction Detection", "H3 NLI Faithfulness"])
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig("results/fig2_method_comparison.pdf")
plt.savefig("results/fig2_method_comparison.png", dpi=300)

print("\nSaved files: results/experiment_results.csv, results/summary_metrics.csv, and PDF/PNG figures.")

In [ ]:
import os
import re
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, CrossEncoder, util
from transformers import AutoTokenizer, AutoModelForCausalLM

# =====================================================================
# 1. FROZEN GATE G3 CONFIGURATION & DEVICE SETUP
# =====================================================================
SEED = 42
N_CLUSTERS = 15  # Full Gate G3 scale
CORPUS_SIZES = [5, 15, 25]
POSITIONS = ["start", "middle", "end"]
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
RETRIEVER_ID = "BAAI/bge-small-en-v1.5"
NLI_ID = "cross-encoder/nli-deberta-v3-small"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

os.makedirs("results", exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Gate G3 Full Run | Device: {device} | N_CLUSTERS={N_CLUSTERS} | K={CORPUS_SIZES}")

# =====================================================================
# 2. LOAD REAL SCIENTIFIC ABSTRACTS (qiaojin/PubMedQA)
# =====================================================================
ds = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")
yes_records = [row for row in ds if row["final_decision"] == "yes" and len(row["context"]["contexts"]) >= 2]
no_records = [row for row in ds if row["final_decision"] == "no" and len(row["context"]["contexts"]) >= 2]
distractor_pool = [
    " ".join(row["context"]["contexts"])[:1100]
    for row in ds
    if row["final_decision"] == "maybe" and len(" ".join(row["context"]["contexts"])) > 400
]

def build_factorial_clusters(n_clusters: int, seed: int = 42):
    rng = random.Random(seed)
    clusters = []
    for idx in range(n_clusters):
        pos_study = yes_records[idx % len(yes_records)]
        neg_study = no_records[idx % len(no_records)]
        topic_question = pos_study["question"]
        ctx_parts = pos_study["context"]["contexts"]
        intro_part = ctx_parts[0][:380]
        result_part = " ".join(ctx_parts[1:])[:420]
        paper_positive = (
            f"Study A ({pos_study['pubid']}): Background and Design: {intro_part} "
            f"Methodological Qualifier: Stratified analysis revealed that treatment efficacy held ONLY in the "
            f"subgroup under 65 years without renal impairment, whereas benefit declined in older or comorbid patients. "
            f"Primary Outcome: {result_part}"
        )
        neg_body = " ".join(neg_study["context"]["contexts"])[:550]
        paper_contradicting = (
            f"Study B ({neg_study['pubid']}): Contradictory Replication: Contrary to positive single-center reports on "
            f"'{topic_question}', this multi-center trial found no statistically significant clinical benefit "
            f"over standard control (p = 0.43) and observed higher adverse-event discontinuation. Details: {neg_body}"
        )
        sampled_distractors = rng.sample(distractor_pool, k=max(CORPUS_SIZES) - 2)
        clusters.append({
            "cluster_id": f"C{idx+1:02d}",
            "question": topic_question,
            "paper_pos": paper_positive,
            "paper_neg": paper_contradicting,
            "distractors": sampled_distractors,
            "h1_keywords": ["under 65", "renal impairment", "subgroup", "only in"],
            "h4_keywords": ["contrary", "contradict", "no statistically significant", "p = 0.43", "however", "conflict", "disagree"],
            "headline_keywords": [w.lower() for w in pos_study["long_answer"].split() if len(w) > 6][:5]
        })
    return clusters

def assemble_prompt_docs(cluster: dict, k: int, position: str):
    sub_distractors = list(cluster["distractors"][:k - 2])
    target_pair = [cluster["paper_pos"], cluster["paper_neg"]]
    if position == "start":
        return target_pair + sub_distractors
    elif position == "end":
        return sub_distractors + target_pair
    elif position == "middle":
        mid = len(sub_distractors) // 2
        return sub_distractors[:mid] + target_pair + sub_distractors[mid:]
    raise ValueError(f"Invalid position: {position}")

# =====================================================================
# 3. INITIALIZE MODELS (SAFEGUARDED AGAINST META TENSOR ERRORS)
# =====================================================================
print(f"Loading Retriever ({RETRIEVER_ID}) and NLI Verifier ({NLI_ID})...")
retriever = SentenceTransformer(RETRIEVER_ID, device=device)
nli_verifier = CrossEncoder(NLI_ID, device=device)

print(f"Loading Generator ({MODEL_ID})...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Fix: Avoid device_map="auto" on CPU to prevent meta tensor offloading errors
if device == "cuda":
    llm = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        device_map="auto"
    )
else:
    llm = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float32
    ).to(device)

llm.eval()

def extract_cited_doc_ids(sentence: str) -> list:
    ids = []
    bracket_groups = re.findall(r'[\[\(](?:Doc\s*)?[\d,\sDoc]+[\]\)]', sentence, flags=re.IGNORECASE)
    for grp in bracket_groups:
        if "doc" in grp.lower():
            ids.extend([int(n) for n in re.findall(r'\d+', grp)])
    return sorted(list(set(ids)))

def strip_citation_tags(sentence: str) -> str:
    return re.sub(r'[\[\(](?:Doc\s*)?[\d,\sDoc]+[\]\)]', '', sentence, flags=re.IGNORECASE).strip()

def call_llm_synthesis(question: str, indexed_docs: list) -> str:
    context_block = "\n\n".join([f"[Doc {doc_num}] {text}" for doc_num, text in indexed_docs])
    messages = [
        {
            "role": "system",
            "content": (
                "You are a rigorous academic synthesis system. Synthesize the findings relevant to the "
                "research question using ONLY the provided documents. Cite every claim with [Doc X]. "
                "Explicitly state any subgroup/methodological qualifiers and report any conflicting results."
            )
        },
        {
            "role": "user",
            "content": f"Documents:\n{context_block}\n\nResearch Question: {question}\n\nAttributed Synthesis:"
        }
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(llm.device)
    with torch.inference_mode():
        out = llm.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    if device == "cuda":
        torch.cuda.empty_cache()
    return text

# =====================================================================
# 4. THREE SYSTEMS: BASELINE 1, BASELINE 2, PROPOSED (FIGURE 2)
# =====================================================================
def run_baseline1_full_context(question: str, docs: list):
    t0 = time.time()
    indexed = [(i + 1, d) for i, d in enumerate(docs)]
    return call_llm_synthesis(question, indexed), time.time() - t0

def run_baseline2_dense_rag(question: str, docs: list, top_k: int = 4):
    t0 = time.time()
    q_emb = retriever.encode(question, convert_to_tensor=True)
    d_embs = retriever.encode(docs, convert_to_tensor=True)
    scores = util.cos_sim(q_emb, d_embs)[0]
    top_idx = torch.topk(scores, k=min(top_k, len(docs))).indices.tolist()
    indexed = [(i + 1, docs[i]) for i in top_idx]
    return call_llm_synthesis(question, indexed), time.time() - t0

def run_proposed_figure2(question: str, docs: list):
    t0 = time.time()
    q_emb = retriever.encode(question, convert_to_tensor=True)
    d_embs = retriever.encode(docs, convert_to_tensor=True)
    scores = util.cos_sim(q_emb, d_embs)[0]
    ranked = torch.argsort(scores, descending=True).tolist()

    left, right = [], []
    for r, idx in enumerate(ranked):
        if r % 2 == 0:
            left.append(idx)
        else:
            right.insert(0, idx)
    reordered = left + right
    indexed = [(i + 1, docs[i]) for i in reordered]
    draft = call_llm_synthesis(question, indexed)

    top1, top2 = ranked[0], ranked[1]
    cited_in_draft = extract_cited_doc_ids(draft)
    clean_draft = strip_citation_tags(draft)
    pairs = [(docs[top1][:900], clean_draft[:450]), (docs[top2][:900], clean_draft[:450])]
    logits = nli_verifier.predict(pairs)
    probs = torch.softmax(torch.tensor(logits), dim=-1)
    max_contradiction = probs[:, 0].max().item()
    missing_top_evidence = ((top1 + 1) not in cited_in_draft) or ((top2 + 1) not in cited_in_draft)

    if missing_top_evidence or max_contradiction >= 0.25:
        focused = [(top1 + 1, docs[top1]), (top2 + 1, docs[top2])]
        final_ans = call_llm_synthesis(question, focused)
        return final_ans, time.time() - t0
    return draft, time.time() - t0

# =====================================================================
# 5. AUTOMATED EVALUATION
# =====================================================================
def score_output(output_text: str, docs: list, cluster: dict) -> dict:
    low = output_text.lower()
    h_hits = sum(1 for kw in cluster["headline_keywords"] if kw in low)
    headline_pct = (h_hits / max(1, len(cluster["headline_keywords"]))) * 100.0
    h1_retained = 100.0 if any(kw in low for kw in cluster["h1_keywords"]) else 0.0
    h4_detected = 100.0 if any(kw in low for kw in cluster["h4_keywords"]) else 0.0

    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', output_text) if s.strip()]
    cited_total, faithful_total = 0, 0
    for sent in sentences:
        cids = extract_cited_doc_ids(sent)
        if cids:
            cited_total += 1
            premises = [docs[c - 1][:800] for c in cids if 1 <= c <= len(docs)]
            clean_hyp = strip_citation_tags(sent)
            if premises and len(clean_hyp) > 10:
                logits = nli_verifier.predict([(" ".join(premises)[:1200], clean_hyp)])
                probs = torch.softmax(torch.tensor(logits), dim=-1)[0]
                p_contra, p_entail = probs[0].item(), probs[1].item()
                if p_entail > p_contra and p_contra < 0.25:
                    faithful_total += 1
    faithfulness_pct = (faithful_total / cited_total * 100.0) if cited_total > 0 else 0.0
    return {
        "headline_recall_pct": round(headline_pct, 2),
        "h1_qualifier_recall_pct": h1_retained,
        "h4_contradiction_pct": h4_detected,
        "h3_nli_faithfulness_pct": round(faithfulness_pct, 2)
    }

# =====================================================================
# 6. RUN FULL GATE G3 SWEEP & EXPORT
# =====================================================================
clusters = build_factorial_clusters(N_CLUSTERS, seed=SEED)
records = []
total_runs = len(clusters) * len(CORPUS_SIZES) * len(POSITIONS) * 3
step = 0

for cluster in clusters:
    for k in CORPUS_SIZES:
        for pos in POSITIONS:
            docs = assemble_prompt_docs(cluster, k, pos)
            for sys_name, fn in [
                ("Baseline_1_FullContext", run_baseline1_full_context),
                ("Baseline_2_DenseRAG", run_baseline2_dense_rag),
                ("Proposed_Fig2_Verified", run_proposed_figure2),
            ]:
                step += 1
                ans, lat = fn(cluster["question"], docs)
                metrics = score_output(ans, docs, cluster)
                records.append({
                    "cluster_id": cluster["cluster_id"],
                    "k_docs": k,
                    "position": pos,
                    "system": sys_name,
                    "latency_sec": round(lat, 2),
                    **metrics,
                    "generated_text": ans
                })
                if step % 27 == 0 or step == total_runs:
                    print(f"Progress: {step}/{total_runs} runs completed...")

df = pd.DataFrame(records)
df.to_csv("results/gate_g3_experiment_results.csv", index=False)
summary = df.groupby(["system", "k_docs", "position"])[
    ["headline_recall_pct", "h1_qualifier_recall_pct", "h4_contradiction_pct", "h3_nli_faithfulness_pct", "latency_sec"]
].mean().round(2).reset_index()
summary.to_csv("results/gate_g3_summary_metrics.csv", index=False)

print("\n=== GATE G3 SUMMARY TABLE ===")
print(summary.to_string(index=False))
print("\nSaved final Gate G3 artifacts in results/!")